# Load a COLIN-processed COOP Amalgamation

One-off load of a coop (TED) created by an amalgamation that was filed and processed **manually in COLIN** (`OTAMA`).

**Phase 1** (the main flow): creates in LEAR the TED business + offices + directors, a COMPLETED paper-only
`amalgamationApplication` filing (`source=COLIN`, with the COLIN event id), the `amalgamations` row
(`amalgamation_type='unknown'`) and one `amalgamating_businesses` row per TING — without touching the TING
business rows.

**Phase 2** (guarded cells at the end, can be run in a later session): links the historic coops — per TING sets
`state_filing_id` + `dissolution_date` (+ HISTORICAL state).

Prerequisites:
- Config: copy `tests/data/.env.sample` to `tests/data/.env` and fill it in
- DB access via the **Cloud SQL connector with IAM auth** (no password)
- API auth: all colin-api and auth-service calls use the `ACCOUNT_SVC_CLIENT_ID` client-credentials token
  (fetched fresh per call via `auth_headers()`); that client must carry the `colin` role
- Writes through the `business_model` models so `sql_versioning` creates the `*_version` rows

## Environment

In [ ]:
# Load config from a .env file
from dotenv import find_dotenv, load_dotenv

ENV_FILE = find_dotenv(usecwd=True)
assert ENV_FILE, 'no .env found — copy tests/data/.env.sample to tests/data/.env and fill it in'
load_dotenv(ENV_FILE, override=True)
print(f'loaded env from {ENV_FILE}')

In [ ]:
import os
from datetime import datetime, timezone

import flask
import requests

from business_model.models import (
    Address,
    AmalgamatingBusiness,
    Amalgamation,
    Business,
    ColinLastUpdate,
    Filing,
    Office,
    Party,
    PartyRole,
)
# not re-exported from business_model.models — import from its module directly
from business_model.models.colin_event_id import ColinEventId
from business_model.models.db import VersioningProxy, db, init_db
from cloud_sql_connector import setup_pg8000_close_event_listener

import legal_api  # CONFIG_MAP lives on the package, not legal_api.config
from legal_api.services.bootstrap import AccountService

# mirrors the first steps of legal_api.create_app() without the service bootstraps
# (LD flags, gcp queue, jwt, endpoints) that a notebook doesn't need
FLASK_APP = flask.Flask(__name__)
FLASK_APP.config.from_object(legal_api.CONFIG_MAP.get(os.getenv('DEPLOYMENT_ENV', 'production'),
                                                      legal_api.CONFIG_MAP['default']))
init_db(FLASK_APP)  # db.init_app + configure_mappers (wires sql_versioning)
FLASK_APP.app_context().push()
if FLASK_APP.config.get('CLOUDSQL_INSTANCE_CONNECTION_NAME'):
    setup_pg8000_close_event_listener(db.engine)

COLIN_API = os.getenv('COLIN_API')
TIMEOUT = 30


def auth_headers() -> dict:
    """Fresh bearer token from the ACCOUNT_SVC client — the service account carries the `colin` role."""
    return {'Authorization': f'Bearer {AccountService.get_bearer_token()}'}

## Inputs — review every value before running

In [ ]:
TED_IDENTIFIER = 'CPXXXXXXX'          # the NEW coop created by the amalgamation (COLIN-assigned)
TING_IDENTIFIERS = ['CPXXXXXXX', 'CPXXXXXXX']  # the amalgamating coops (already in LEAR, HISTORICAL)
OTAMA_EVENT_ID = 107851387                 # the COLIN event id of the manual amalgamation; None = discover from events below
EFFECTIVE_DATE_OVERRIDE = ''        # ISO string to override the COLIN founding/event date, e.g. '2026-10-01T07:00:00+00:00'
COURT_APPROVAL = False
ASSOCIATION_TYPE = 'CP'               # CP | HC (housing) | CSC (community service)
CERTIFIED_BY = 'COLIN (manual filing)'
ACCOUNT_ID = None                     # auth org id to affiliate the new coop to; None = skip affiliation

TED_IDENTIFIER, TING_IDENTIFIERS

## Safety checks — BAIL if the TED already exists

In [ ]:
ted = Business.find_by_identifier(TED_IDENTIFIER)
assert not ted, f'{TED_IDENTIFIER} already exists in LEAR — DO NOT load it again'

tings = {}
for ting_identifier in TING_IDENTIFIERS:
    ting = Business.find_by_identifier(ting_identifier)
    assert ting, f'TING {ting_identifier} not found in LEAR — check the identifier'
    tings[ting_identifier] = ting
    if ting.state != Business.State.HISTORICAL:
        print(f'WARNING: TING {ting_identifier} is {ting.state} (expected HISTORICAL)')
    if ting.state_filing_id:
        print(f'WARNING: TING {ting_identifier} already has state_filing_id={ting.state_filing_id}')

{k: (v.state.name if v.state else None, v.state_filing_id) for k, v in tings.items()}

## Fetch the COLIN snapshot and the OTAMA event

In [ ]:
r = requests.get(f'{COLIN_API}/{TED_IDENTIFIER}/snapshot', headers=auth_headers(), timeout=TIMEOUT)
assert r.status_code == 200, f'snapshot failed: {r.status_code} {r.text}'
snapshot = r.json()
snapshot['business']

In [ ]:
r = requests.get(f'{COLIN_API}/event/corp_num/{TED_IDENTIFIER}', headers=auth_headers(), timeout=TIMEOUT)
events = r.json().get('events', []) if r.status_code == 200 else []
otama_events = [e for e in events if e.get('filing_typ_cd') == 'OTAMA']
print(f'{len(events)} events for {TED_IDENTIFIER}; OTAMA events: {otama_events}')

if OTAMA_EVENT_ID is None:
    assert len(otama_events) == 1, 'could not uniquely identify the OTAMA event — set OTAMA_EVENT_ID manually'
    OTAMA_EVENT_ID = otama_events[0]['event_id']
OTAMA_EVENT_ID

In [ ]:
effective_date = datetime.fromisoformat(EFFECTIVE_DATE_OVERRIDE or snapshot['business']['foundingDate'])
if effective_date.tzinfo is None:
    effective_date = effective_date.replace(tzinfo=timezone.utc)

print('legal name :', snapshot['business']['legalName'])
print('effective  :', effective_date)
print('offices    :', list(snapshot['offices'].keys()))
print('directors  :', [p['officer'] for p in snapshot['parties']])
assert snapshot['business']['legalType'] == 'CP', 'snapshot is not a coop — stop'

## Build everything in one transaction

Mirrors the filer's `amalgamation_application.process()` and data-tool's `load_amalgamation_snapshot`,
minus the TING updates (Phase 2) and with the COLIN identifier instead of `get_next_corp_num`.

In [ ]:
def build_address(address_info: dict, address_type: str) -> Address:
    """Snapshot addresses are already LEAR-shaped with alpha-2 country codes."""
    return Address(
        street=address_info.get('streetAddress') or '',
        street_additional=address_info.get('streetAddressAdditional') or '',
        city=address_info.get('addressCity') or '',
        region=address_info.get('addressRegion') or '',
        country=address_info.get('addressCountry') or 'CA',
        postal_code=address_info.get('postalCode') or '',
        delivery_instructions=address_info.get('deliveryInstructions') or '',
        address_type=address_type.replace('Address', ''),
    )


def build_office(business: Business, office_type: str, addresses: dict) -> Office:
    office = Office()
    office.office_type = office_type
    office.addresses = []
    for key, addr in addresses.items():
        if addr:
            office.addresses.append(build_address(addr, key))
    business.offices.append(office)
    return office


def build_party_roles(business: Business, party_info: dict) -> list[PartyRole]:
    officer = party_info['officer']
    party = Party(
        first_name=(officer.get('firstName') or '').upper(),
        last_name=(officer.get('lastName') or '').upper(),
        middle_initial=(officer.get('middleInitial') or officer.get('middleName') or '').upper(),
        organization_name=(officer.get('organizationName') or '').upper(),
        party_type=officer.get('partyType') or 'person',
    )
    if party_info.get('deliveryAddress'):
        party.delivery_address = build_address(party_info['deliveryAddress'], Address.DELIVERY)
    if party_info.get('mailingAddress'):
        party.mailing_address = build_address(party_info['mailingAddress'], Address.MAILING)
    roles = []
    for role in party_info.get('roles', []):
        role_type = (role.get('roleType') or '').lower()
        if role_type != PartyRole.RoleTypes.DIRECTOR.value:
            print(f"skipping non-director role: {role}")
            continue
        party_role = PartyRole(
            role=role_type,
            appointment_date=role.get('appointmentDate'),
            cessation_date=role.get('cessationDate'),
            party=party,
        )
        business.party_roles.append(party_role)  # dynamic relationship: append only, no extend
        roles.append(party_role)
    return roles

In [ ]:
try:
    transaction_id = VersioningProxy.get_transaction_id(db.session())

    # --- TED business ---
    business = Business(
        identifier=TED_IDENTIFIER,              # COLIN-assigned — never get_next_corp_num
        legal_name=snapshot['business']['legalName'],
        legal_type=Business.LegalTypes.COOP.value,
        association_type=ASSOCIATION_TYPE,
        founding_date=effective_date,
        last_coa_date=effective_date,
        last_cod_date=effective_date,
        state=Business.State.ACTIVE,
        tax_id=snapshot['business'].get('taxId'),
    )
    db.session.add(business)

    for office_type, addresses in snapshot['offices'].items():
        build_office(business, office_type, addresses)
    for party_info in snapshot['parties']:
        build_party_roles(business, party_info)

    # --- the amalgamation filing (ledger placeholder, paper only) ---
    effective_iso = effective_date.isoformat()
    filing_json = {
        'filing': {
            'header': {
                'name': 'amalgamationApplication',
                'date': effective_iso[:10],
                'certifiedBy': CERTIFIED_BY,
            },
            # 'type' is required: the filing_json setter derives filing_sub_type
            # by direct index (FILING_SUB_TYPE_KEYS['amalgamationApplication'])
            'amalgamationApplication': {'type': Amalgamation.AmalgamationTypes.unknown.name},
            'business': {
                'identifier': TED_IDENTIFIER,
                'legalType': Business.LegalTypes.COOP.value,
                'foundingDate': effective_iso,
            },
        },
    }
    filing = Filing(effective_date=effective_date, filing_json=filing_json)
    filing.skip_status_listener = True
    filing._status = Filing.Status.COMPLETED.value
    filing.source = Filing.Source.COLIN.value
    filing.paper_only = True
    filing.filing_date = effective_date
    filing._completion_date = effective_date
    filing.transaction_id = transaction_id
    filing._meta_data = {
        'colinFilingInfo': {'filingType': 'OTAMA', 'eventId': OTAMA_EVENT_ID},
        'isLedgerPlaceholder': True,
        'colinDisplayName': 'Amalgamation Application',
    }
    colin_event = ColinEventId()
    colin_event.colin_event_id = OTAMA_EVENT_ID
    filing.colin_event_ids.append(colin_event)
    db.session.add(filing)

    db.session.flush()
    filing.business_id = business.id   # mandatory — NULL 500s every linked-TING GET
    for office in business.offices.all():
        for address in office.addresses.all():
            address.business_id = business.id   # the filer's create_office does the same

    # --- amalgamation + amalgamating businesses (NO TING business-row updates) ---
    amalgamation = Amalgamation()
    amalgamation.business_id = business.id
    amalgamation.filing_id = filing.id
    amalgamation.amalgamation_date = effective_date
    amalgamation.amalgamation_type = Amalgamation.AmalgamationTypes.unknown.name
    amalgamation.court_approval = COURT_APPROVAL
    for ting_identifier, ting in tings.items():
        amalgamating = AmalgamatingBusiness()
        amalgamating.role = AmalgamatingBusiness.Role.amalgamating.name
        amalgamating.business_id = ting.id
        amalgamation.amalgamating_businesses.append(amalgamating)
    db.session.add(amalgamation)

    db.session.commit()
    print(f'COMMITTED  business_id={business.id}  filing_id={filing.id}  amalgamation_id={amalgamation.id}')
except Exception:
    db.session.rollback()
    raise

## Post-load checks (model level)

In [ ]:
loaded = Business.find_by_identifier(TED_IDENTIFIER)
assert loaded and loaded.state == Business.State.ACTIVE
assert loaded.amalgamation.one_or_none(), 'amalgamations row missing'
assert loaded.is_tombstone is False, 'TED must not read as tombstone'

the_filing = Filing.find_by_id(filing.id)
assert the_filing.status == Filing.Status.COMPLETED.value
assert the_filing.paper_only and the_filing.source == Filing.Source.COLIN.value
assert the_filing.business_id == loaded.id
assert the_filing.colin_event_ids, 'colin_event_ids row missing'

# must NOT be picked up by the LEAR->COLIN sync job
pending_for_colin = Filing.get_completed_filings_for_colin(limit=1000)
assert the_filing.id not in [f.id for f in pending_for_colin], 'filing would sync back into COLIN!'
loaded.json() if callable(getattr(loaded, 'json', None)) else 'ok'

## Auth: create the entity (and optionally affiliate)

In [ ]:
r = requests.post(
    os.getenv('ACCOUNT_SVC_ENTITY_URL'),
    headers=auth_headers(),
    json={
        'businessIdentifier': loaded.identifier,
        'corpTypeCode': loaded.legal_type,
        'name': loaded.legal_name,
    },
    timeout=TIMEOUT,
)
r.status_code, r.text

In [ ]:
if ACCOUNT_ID:
    r = requests.post(
        os.getenv('ACCOUNT_SVC_AFFILIATE_URL').format(account_id=ACCOUNT_ID),
        headers=auth_headers(),
        json={'businessIdentifier': loaded.identifier},
        timeout=TIMEOUT,
    )
    print(r.status_code, r.text)
else:
    print('ACCOUNT_ID not set — skipping affiliation')

## Phase 2: link the historic coops (TINGs)

Per TING this sets exactly what the filer's `dissolve_amalgamating_business` sets — `state=HISTORICAL`, `dissolution_date` = amalgamation date, `state_filing_id` = the TED's amalgamation filing — and nothing else (no party cessation, no TING ledger row; that matches real filer behaviour). Unlocks `amalgamatedInto` on the TING dashboards, the "Amalgamated" business-summary entry, and the restoration blocker.

In [ ]:
RUN_TING_UPDATES = False   # Phase 2 gate — set True when ready to link the historic coops

# re-resolve from the DB so this cell works standalone in a later session
ted_business = Business.find_by_identifier(TED_IDENTIFIER)
assert ted_business, f'{TED_IDENTIFIER} not in LEAR — run Phase 1 first'
ted_amalgamation = ted_business.amalgamation.one_or_none()
assert ted_amalgamation, 'amalgamations row missing — run Phase 1 first'
amalgamation_filing = Filing.find_by_id(ted_amalgamation.filing_id)
assert amalgamation_filing and amalgamation_filing.business_id == ted_business.id

linked_ting_ids = {ab.business_id for ab in ted_amalgamation.amalgamating_businesses}

if RUN_TING_UPDATES:
    try:
        VersioningProxy.get_transaction_id(db.session())  # version the TING updates in their own transaction
        for ting_identifier in TING_IDENTIFIERS:
            ting = Business.find_by_identifier(ting_identifier)
            assert ting, f'TING {ting_identifier} not found'
            assert ting.id in linked_ting_ids, \
                f'TING {ting_identifier} has no amalgamating_businesses row — fix Phase 1 first'
            if ting.state_filing_id and ting.state_filing_id != amalgamation_filing.id:
                print(f'SKIPPING {ting_identifier}: unexpected state_filing_id={ting.state_filing_id}')
                continue
            # mirrors business-filer amalgamation_application.dissolve_amalgamating_business
            ting.state = Business.State.HISTORICAL
            ting.dissolution_date = amalgamation_filing.effective_date
            ting.state_filing_id = amalgamation_filing.id
            db.session.add(ting)
            print(f'{ting_identifier}: HISTORICAL, dissolution_date={ting.dissolution_date}, '
                  f'state_filing_id={ting.state_filing_id}')
        db.session.commit()
        print('TING updates COMMITTED')
    except Exception:
        db.session.rollback()
        raise
else:
    print('RUN_TING_UPDATES is False — nothing changed')

In [ ]:
# Phase 2 post-checks: every TING now resolves amalgamatedInto -> the TED
if RUN_TING_UPDATES:
    for ting_identifier in TING_IDENTIFIERS:
        ting = Business.find_by_identifier(ting_identifier)
        assert ting.state == Business.State.HISTORICAL
        assert ting.state_filing_id == amalgamation_filing.id
        amalgamated_into = ting.get_amalgamated_into()
        assert amalgamated_into, f'{ting_identifier}: amalgamatedInto did not resolve'
        assert amalgamated_into.get('identifier') == TED_IDENTIFIER
        print(f'{ting_identifier}: amalgamatedInto -> {amalgamated_into}')
else:
    print('RUN_TING_UPDATES is False — nothing to check')